#### Text Feature Engineering Assignment — Complete Implementation

#### Prepare the data -> Setup and Load reviews.csv

In [49]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [5]:
# Path to static dataset
csv_path = "reviews.csv"
df = pd.read_csv(csv_path)
df = df.dropna(subset=["review_text"]).copy()
df["review_text"] = df["review_text"].astype(str)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (500, 1)


,review_text
0,Terrible build quality i feel
1,Worth the money
2,Superb quality
3,Product stopped working
4,Not worth the price


### Task 1) Preprocessing

In [36]:
# Task 1: Preprocessing

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=True, apply_lemmatization=True):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    
    # 3. Remove digits
    text = re.sub(r"\d+", " ", text)
    
    # 4. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # 5. Tokenization
    tokens = text.split()
    
    # 6. Optional stopword removal
    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]
    
    # 7. Optional lemmatization
    if apply_lemmatization:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

# Apply preprocessing
df['cleaned_review'] = df['review_text'].apply(preprocess_text)

print(df[['review_text', 'cleaned_review']].head())
print(df['cleaned_review'].size)

                     review_text               cleaned_review
0  Terrible build quality i feel  terrible build quality feel
1                Worth the money                  worth money
2                 Superb quality               superb quality
3        Product stopped working      product stopped working
4            Not worth the price                  worth price
500


### Task 2) Vocabulary Creation

In [17]:
# Task 2: Vocabulary Creation

all_words = " ".join(df['cleaned_review']).split()
word_freq = Counter(all_words)

# Vocabulary
vocab = sorted(set(all_words))

print("Vocabulary Size:", len(vocab))

print("\nTop 20 Frequent Words:")
for word, freq in word_freq.most_common(20):
    print(f"{word}: {freq}")

Vocabulary Size: 51

Top 20 Frequent Words:
product: 80
quality: 79
experience: 51
price: 50
work: 40
expected: 40
loved: 38
bad: 37
performance: 34
okay: 32
worth: 31
good: 31
money: 27
purchase: 27
fine: 25
could: 25
better: 25
excellent: 24
fast: 24
delivery: 24


### Task 3: Feature Engineering

#### Task 3A: One Hot Encoding (Document-Level Vector)

In [47]:


# To avoid extremely huge vectors, optionally limit vocabulary
top_n_vocab = [word for word, freq in word_freq.most_common(1000)]
word_to_index = {word: idx for idx, word in enumerate(top_n_vocab)}

def one_hot_encode_document(text, word_to_index):
    # print("text:::",text,":::word_to_index::::",word_to_index)
    vector = np.zeros(len(word_to_index), dtype=int)
    words = set(text.split())  # document-level presence
    for word in words:
        if word in word_to_index:
            vector[word_to_index[word]] = 1
    return vector

ohe_matrix = np.array([one_hot_encode_document(text, word_to_index) for text in df['cleaned_review']])

print("OHE matrix shape:", ohe_matrix.shape)
print("\nSample OHE vector:")
print(ohe_matrix[0])

OHE matrix shape: (500, 51)

Sample OHE vector:
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 1 1 0 0 0 0 0 1]


#### Task 3B: Bag of Words using CountVectorizer

In [48]:
bow_vectorizer = CountVectorizer(max_features=1000)
X_bow = bow_vectorizer.fit_transform(df['cleaned_review'])

print("BoW matrix shape:", X_bow.shape)

# Vocabulary from BoW
bow_vocab = bow_vectorizer.get_feature_names_out()
print("\nSample BoW vocabulary words:")
print(bow_vocab)

BoW matrix shape: (500, 51)

Sample BoW vocabulary words:
['amazing' 'average' 'bad' 'better' 'build' 'buy' 'could' 'decent'
 'delivery' 'described' 'design' 'disappointed' 'ever' 'excellent'
 'expected' 'experience' 'fast' 'feel' 'fine' 'five' 'good' 'great'
 'highly' 'loved' 'moderate' 'money' 'neither' 'nothing' 'okay'
 'perfectly' 'performance' 'poor' 'price' 'product' 'purchase' 'quality'
 'really' 'recommend' 'satisfactory' 'satisfied' 'special' 'star'
 'stopped' 'superb' 'terrible' 'value' 'waste' 'work' 'working' 'worst'
 'worth']


#### Task 3C: TF-IDF using TfidfVectorizer

In [50]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_review'])

print("TF-IDF matrix shape:", X_tfidf.shape)

tfidf_vocab = tfidf_vectorizer.get_feature_names_out()
print("\nSample TF-IDF vocabulary words:")
print(tfidf_vocab)

TF-IDF matrix shape: (500, 51)

Sample TF-IDF vocabulary words:
['amazing' 'average' 'bad' 'better' 'build' 'buy' 'could' 'decent'
 'delivery' 'described' 'design' 'disappointed' 'ever' 'excellent'
 'expected' 'experience' 'fast' 'feel' 'fine' 'five' 'good' 'great'
 'highly' 'loved' 'moderate' 'money' 'neither' 'nothing' 'okay'
 'perfectly' 'performance' 'poor' 'price' 'product' 'purchase' 'quality'
 'really' 'recommend' 'satisfactory' 'satisfied' 'special' 'star'
 'stopped' 'superb' 'terrible' 'value' 'waste' 'work' 'working' 'worst'
 'worth']


### Task 4: Comparison Analysis

In [60]:

comparison_df = pd.DataFrame({
    "Technique": ["One Hot Encoding", "Bag of Words", "TF-IDF"],
    "Meaning": [
        "Marks whether a word is present or absent in a document",
        "Counts how many times each word appears in a document",
        "Measures word importance based on frequency in document and rarity across documents"
    ],
    "Captures Frequency": ["No", "Yes", "Yes"],
    "Captures Importance": ["No", "Partially", "Yes"],
    "Matrix Type": ["Dense", "Sparse", "Sparse"],
    "Best Use": [
        "Simple presence/absence features",
        "Basic text classification",
        "Search, ranking, and improved text classification"
    ]
})

print(comparison_df)

          Technique                                            Meaning  \
0  One Hot Encoding  Marks whether a word is present or absent in a...   
1      Bag of Words  Counts how many times each word appears in a d...   
2            TF-IDF  Measures word importance based on frequency in...   

  Captures Frequency Captures Importance Matrix Type  \
0                 No                  No       Dense   
1                Yes           Partially      Sparse   
2                Yes                 Yes      Sparse   

                                            Best Use  
0                   Simple presence/absence features  
1                          Basic text classification  
2  Search, ranking, and improved text classification  


| Technique | Captures Frequency | Captures Importance | Use Case                 |
| --------- | ------------------ | ------------------- | ------------------------ |
| OHE       | ❌ No               | ❌ No                | Presence only            |
| BoW       | ✅ Yes              | ⚠️ Partial          | Basic ML                 |
| TF-IDF    | ✅ Yes              | ✅ Yes               | Important word detection |


In [52]:
# Get feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

# Take one document
doc_index = 0

# Convert to array
tfidf_scores = X_tfidf[doc_index].toarray().flatten()

# Get top important words
top_indices = tfidf_scores.argsort()[::-1][:10]

print("Top Important Words (TF-IDF):\n")
for idx in top_indices:
    if tfidf_scores[idx] > 0:
        print(f"{feature_names[idx]} : {tfidf_scores[idx]:.4f}")

Top Important Words (TF-IDF):

feel : 0.6733
build : 0.4801
terrible : 0.4801
quality : 0.2926


#### Task 5: Sparse Matrix Analysis

In [59]:
def calculate_sparsity(matrix):
    if hasattr(matrix, "toarray"):  # sparse matrix
        total_elements = matrix.shape[0] * matrix.shape[1]
        non_zero = matrix.count_nonzero()
    else:  # dense numpy array
        total_elements = matrix.size
        non_zero = np.count_nonzero(matrix)
    
    zero_elements = total_elements - non_zero
    sparsity = (zero_elements / total_elements) * 100
    return total_elements, non_zero, zero_elements, sparsity

matrices = {
    "OHE": ohe_matrix,
    "BoW": X_bow,
    "TF-IDF": X_tfidf
}

for name, matrix in matrices.items():
    total, non_zero, zero, sparsity = calculate_sparsity(matrix)
    print(f"\n{name} Matrix")
    print("Shape:", matrix.shape)
    print("Total Elements:", total)
    print("Non-Zero Elements:", non_zero)
    print("Zero Elements:", zero)
    print(f"Sparsity: {sparsity:.2f}%")


OHE Matrix
Shape: (500, 51)
Total Elements: 25500
Non-Zero Elements: 1230
Zero Elements: 24270
Sparsity: 95.18%

BoW Matrix
Shape: (500, 51)
Total Elements: 25500
Non-Zero Elements: 1230
Zero Elements: 24270
Sparsity: 95.18%

TF-IDF Matrix
Shape: (500, 51)
Total Elements: 25500
Non-Zero Elements: 1230
Zero Elements: 24270
Sparsity: 95.18%


#####

Text feature matrices are sparse because each document contains only a small subset of the total vocabulary, resulting in many zero values, which increases memory usage and reduces computational efficiency in large-scale systems.

#####

All three representations (OHE, BoW, TF-IDF) produce highly sparse matrices because each document contains only a small subset of the total vocabulary. 
This results in a large number of zero values. Sparse matrices are inefficient in large-scale systems because they consume unnecessary memory and increase computational overhead. 
To address this, optimized sparse storage formats such as CSR are used in real-world applications.

### Task 6: Real-world Questions

#####
1 Why Bag of Words fails in understanding semantic meaning (example: similar meaning words)
2 When to use Bag of Words and TF-IDF in industry
3 Limitations of TF-IDF in real applications

In [61]:
# Task 6: Real-world Questions

real_world_answers = {
    "Why BoW fails in semantics":
        "Bag of Words cannot understand meaning or context. "
        "For example, 'good' and 'excellent' may have similar meaning, "
        "but BoW treats them as completely different words.",

    "When to use BoW and TF-IDF":
        "Bag of Words is useful in simple baseline text classification problems. "
        "TF-IDF is used when word importance matters, such as document ranking, "
        "search engines, spam detection, and sentiment analysis.",

    "Limitations of TF-IDF":
        "TF-IDF does not capture word order, semantics, or context. "
        "It also cannot understand sarcasm, negation properly, or synonyms."
}

for question, answer in real_world_answers.items():
    print(f"\n{question}:\n{answer}")


Why BoW fails in semantics:
Bag of Words cannot understand meaning or context. For example, 'good' and 'excellent' may have similar meaning, but BoW treats them as completely different words.

When to use BoW and TF-IDF:
Bag of Words is useful in simple baseline text classification problems. TF-IDF is used when word importance matters, such as document ranking, search engines, spam detection, and sentiment analysis.

Limitations of TF-IDF:
TF-IDF does not capture word order, semantics, or context. It also cannot understand sarcasm, negation properly, or synonyms.
